# Step 1: Import Required Packages

In [1]:
import os

from dotenv import load_dotenv

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma

from langchain_core.documents import Document

from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    SystemMessage
)

C:\Users\Rohit singh\AppData\Local\Temp\ipykernel_9100\3163036107.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# Step 2: Load PDF

In [2]:
# Step 2: Load PDF

pdf_path = "data/Python_Notes.pdf"
loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("PDF loaded successfully")
print("Total pages:", len(documents))

PDF loaded successfully
Total pages: 61


# Step 3: Split Documents into Chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Documents split into chunks successfully")
print("Total chunks:", len(chunks))

Documents split into chunks successfully
Total chunks: 107


In [4]:
print(chunks[0].page_content)

1 
 
PREFACE  
Welcome to the “Ultimate Python Programming Handbook," your comprehensive guide to 
mastering Python programming. This handbook is designed for beginners and anyone looking to 
strengthen their foundational knowledge of Python, a versatile and user-friendly programming 
language. 
PURPOSE AND AUDIENCE  
This handbook aims to make programming accessible and enjoyable for everyone. Whether 
you're a student new to coding, a professional seeking to enhance your skills, or an enthusiast 
exploring Python, this handbook will definitely be helpful. Python's simplicity and readability 
make it an ideal starting point for anyone interested in programming. 
STRUCTURE AND CONTENT  
The handbook is divided into clear, concise chapters, each focused on a specific aspect of 
Python: 
• Fundamental Concepts: Start with the basics, such as installing Python and writing your 
first program. 
• Practical Examples: Illustrative examples and sample code demonstrate the


In [5]:
print(chunks[0].metadata)

{'producer': '3.0.10 (5.0.18)', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-06-16T02:12:11+05:30', 'author': 'Harry Codes', 'moddate': '2024-06-15T22:44:10+02:00', 'source': 'data/Python_Notes.pdf', 'total_pages': 61, 'page': 1, 'page_label': '2'}


# Step 4: Create Embeddings

In [6]:
load_dotenv()

google_api_key = os.getenv("GEMINI_API_KEY")

if not google_api_key:
    print("GEMINI_API_KEY not found")
else:
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=google_api_key
    )

    print("Embedding model initialized successfully")

Embedding model initialized successfully


In [7]:
# Test Embedding

test_text = "What is machine learning?"

vector = embeddings.embed_query(test_text)

print("Vector dimensions:", len(vector))
print("First 10 values:", vector[:10])

Vector dimensions: 3072
First 10 values: [-0.02689046, 0.012816175, -0.009027316, -0.058475167, -0.019788653, 0.010473229, -0.007066461, 0.0018563827, 0.012909309, 0.041982897]


# Step 5: Vector Database

In [11]:
from time import sleep

vector_store = Chroma(
    collection_name="rag_documents",
    embedding_function=embeddings,
    persist_directory="chroma_db"
)

batch_size = 10

for i in range(0, len(chunks), batch_size):

    batch = chunks[i:i + batch_size]

    vector_store.add_documents(batch)

    stored = min(i + batch_size, len(chunks))

    print(f"Stored {stored}/{len(chunks)} chunks")

    sleep(10)

print("All documents stored successfully")

Stored 10/107 chunks
Stored 20/107 chunks
Stored 30/107 chunks
Stored 40/107 chunks
Stored 50/107 chunks
Stored 60/107 chunks
Stored 70/107 chunks
Stored 80/107 chunks
Stored 90/107 chunks
Stored 100/107 chunks
Stored 107/107 chunks
All documents stored successfully


In [12]:
# import shutil
# import os

# if os.path.exists("chroma_db"):
#     shutil.rmtree("chroma_db")
#     print("Old Chroma DB deleted")

# print("Ready for fresh database")

In [13]:
print("Total chunks:", len(chunks))
print("Chroma documents:", vector_store._collection.count())

Total chunks: 107
Chroma documents: 107


# Step 6: Create Retriever

In [10]:
# Create and Test Retriever

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Test Retriever
query = "What is machine learning?"

relevant_docs = retriever.invoke(query)

print("Relevant chunks found:", len(relevant_docs))

for i, doc in enumerate(relevant_docs, start=1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content[:500])
    print("Metadata:", doc.metadata)

NameError: name 'vector_store' is not defined

# Step 7: Retrieve Relevant Content

In [ ]:
# Retrieve Relevant Content

query = "What is machine learning?"

relevant_docs = retriever.invoke(query)

context = "\n\n".join(
    doc.page_content
    for doc in relevant_docs
)

print("Relevant context retrieved successfully")

print("\n--- Retrieved Context ---\n")
print(context)

print("\n--- Sources ---")

for i, doc in enumerate(relevant_docs, start=1):
    print(
        f"Chunk {i} → "
        f"Page: {doc.metadata.get('page')}, "
        f"Source: {doc.metadata.get('source')}"
    )

# Step 8: Initialize Gemini LLM

In [ ]:
load_dotenv()

google_api_key = os.getenv("GEMINI_API_KEY")

if not google_api_key:
    print("GEMINI_API_KEY not found")
else:
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0.2,
        google_api_key=google_api_key
    )

In [ ]:
# Test Gemini

response = llm.invoke("What is Python?")
print(response.content)

# Step 9: Create RAG Chain

In [ ]:
# Step 9: Create RAG Chain

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the user's question using ONLY the context provided below.

If the answer is not available in the context, say:
"I could not find the answer in the provided PDF."

Context:
{context}

Question:
{question}

Answer:
""")

def rag_chain(question):

    # 1. Retrieve relevant documents
    relevant_docs = retriever.invoke(question)

    # 2. Combine retrieved chunks
    context = "\n\n".join(
        doc.page_content
        for doc in relevant_docs
    )

    # 3. Create prompt
    formatted_prompt = prompt.format(
        context=context,
        question=question
    )

    # 4. Ask Gemini
    response = llm.invoke(formatted_prompt)

    # 5. Extract text from Gemini response
    if isinstance(response.content, list):
        answer = "\n".join(
            item["text"]
            for item in response.content
            if isinstance(item, dict) and item.get("type") == "text"
        )
    else:
        answer = response.content

    return answer

In [ ]:
question = "What is Python?"

answer = rag_chain(question)
print("AI:", answer)

# Step 10 — Final RAG Chatbot

In [ ]:
# Final RAG Chatbot

def ask_rag(question):
    relevant_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in relevant_docs
    )

    formatted_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)

    # Extract text from Gemini response
    if isinstance(response.content, list):
        answer = "\n".join(
            item["text"]
            for item in response.content
            if isinstance(item, dict) and item.get("type") == "text"
        )
    else:
        answer = response.content

    return answer

In [ ]:
question = "What is Python?"

answer = ask_rag(question)
print("\nAI:", answer)

In [ ]:
# Final RAG Chatbot

print("RAG AI Chatbot 🤖")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("AI: Goodbye! 👋")
        break

    answer = ask_rag(user_input)

    print("\nAI:", answer)
    print()

In [ ]:
print("Total chunks:", len(chunks))
print("Batch size:", batch_size)


In [ ]:
print(vector_store._collection.count())